In [ ]:
import os
import torch

from google.colab import userdata
access_token = userdata.get("HF_TOKEN")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

In [ ]:
from transformers import AutoTokenizer

tokenizer_name = "bigcode/starcoder2-3b"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, token=access_token)

model_config = {
    "tokenizer_name": tokenizer_name,
    "batch_size": 8,
    "context_length": 512,
    "vocab_size": len(tokenizer),
    "embedding_dim": 256,
    "n_layers": 4,
    "n_heads": 4,
    "n_kv_heads": 2,
}

batch_size = model_config["batch_size"]
context_length = model_config["context_length"]
vocab_size = model_config["vocab_size"]
embedding_dim = model_config["embedding_dim"]
n_layers = model_config["n_layers"]
n_heads = model_config["n_heads"]
n_kv_heads = model_config["n_kv_heads"]

In [ ]:
from google.colab import drive
import sys

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/tiny-coder')

In [ ]:
from torch.utils.data import DataLoader
from data import create_python_dataset, CoderDataset

train_raw = create_python_dataset("/content/drive/MyDrive/tiny-coder/train.jsonl").shuffle(seed=42, buffer_size=10000)
val_raw = create_python_dataset("/content/drive/MyDrive/tiny-coder/val.jsonl")

train_set = CoderDataset(train_raw, tokenizer, context_length)
val_set = CoderDataset(val_raw, tokenizer, context_length)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
def generate_tokens(input, num_tokens, model):
    encoded = tokenizer.encode(input, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        for _ in range(num_tokens):
            model_input = encoded[:, -context_length:]
            output = model(model_input)
            next_token_pred = torch.argmax(output[:, -1, :], dim=-1).unsqueeze(1)
            encoded = torch.cat((encoded, next_token_pred), dim=1)
    return encoded

In [ ]:
def evaluate_model(data_loader, model, loss_fn):
    model.eval()
    losses = []

    with torch.no_grad():
        for x, y in iter(data_loader):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(torch.flatten(logits, end_dim=1), torch.flatten(y))
            losses.append(loss.item())

    return sum(losses) / len(losses)

In [ ]:
def train_model(train_loader, val_loader, model, optimizer, loss_fn, epochs=10, eval_interval=100, scheduler=None):
    model.train()
    train_losses = []
    val_losses = []
    global_step = 0

    for epoch in range(1, epochs + 1):
        for x, y in train_loader:
            global_step += 1
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            logits = model(x)
            loss = loss_fn(torch.flatten(logits, end_dim=1), torch.flatten(y))
            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            train_losses.append(loss.item())

            if global_step % eval_interval == 0:
                val_loss = evaluate_model(val_loader, model, loss_fn)
                val_losses.append((global_step, val_loss))
                print(
                    f"Epoch {epoch} | Step {global_step}: "
                    f"Train loss = {loss.item():.4f} "
                    f"Val loss = {val_loss:.4f}"
                )
                model.train()

    return train_losses, val_losses, global_step

In [ ]:
from model import LLM
from torch import nn, optim

model = LLM(
    vocab_size,
    context_length,
    embedding_dim,
    n_layers,
    n_heads,
    n_kv_heads,
    intermediate_dim=embedding_dim * 4,
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
from transformers import get_cosine_schedule_with_warmup

epochs = 3

# Since CoderDataset is an IterableDataset, it doesn't have a len().
# We count the batches manually.
print("Counting batches in train_loader to calculate total steps...")
batches_per_epoch = sum(1 for _ in train_loader)
total_steps = batches_per_epoch * epochs
warmup_steps = int(0.1 * total_steps) # 10% warmup

# Setup the scheduler: Linear warmup with cosine decay
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

initial_val_loss = evaluate_model(val_loader, model, loss_fn)
print(f"Initial val loss: {initial_val_loss:.4f}")


train_losses, val_losses, completed_steps = train_model(
    train_loader,
    val_loader,
    model,
    optimizer,
    loss_fn,
    epochs=epochs,
    eval_interval=250,
    scheduler=scheduler
)

final_val_loss = evaluate_model(val_loader, model, loss_fn)

print(f"Final val loss: {final_val_loss:.4f}")
print(f"Completed steps: {completed_steps}")

In [ ]:
prompts = [
    "def fibonacci(n):",
    "def read_json_file(path):",
    "class Stack:",
]

for prompt in prompts:
  decoded = tokenizer.decode(generate_tokens(prompt, 100, model)[0])
  print(decoded)

In [ ]:
from pathlib import Path

Path("checkpoints").mkdir(exist_ok=True)
checkpoint_path = f"/content/drive/MyDrive/tiny-coder/checkpoints/base_model_{completed_steps}steps.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": completed_steps,
        "model_config": model_config,
        "final_val_loss": final_val_loss,
    },
    checkpoint_path,
)

checkpoint_path

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
completed_steps = checkpoint["step"]

completed_steps